<a href="https://colab.research.google.com/github/arturbernardo/tse_data/blob/main/tse_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
import math


drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import pandas as pd

In [6]:
pd.options.display.max_columns = None

In [7]:

from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

import pandas as pd
from getpass import getpass
from sqlalchemy import create_engine
from sqlalchemy import text

In [8]:
!wget -U "Mozilla" -O municipios_ibge_tse.csv https://raw.githubusercontent.com/arturbernardo/Municipios-Brasileiros-TSE/refs/heads/master/municipios_brasileiros_tse.csv

--2025-05-01 20:43:41--  https://raw.githubusercontent.com/arturbernardo/Municipios-Brasileiros-TSE/refs/heads/master/municipios_brasileiros_tse.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 178410 (174K) [text/plain]
Saving to: ‘municipios_ibge_tse.csv’

municipios_ibge_tse 100%[===================>] 174.23K  --.-KB/s    in 0.03s   

2025-05-01 20:43:41 (5.22 MB/s) - ‘municipios_ibge_tse.csv’ saved [178410/178410]



In [8]:
port = 5432
database = "postgres"
user = "postgres"
host = "34.95.206.58"
password = getpass("Digite sua senha: ")
database = "postgres"

engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}")

with engine.connect() as connection:
    result = connection.execute(text("SELECT version()"))
    for row in result:
        print(row)


Digite sua senha: ··········
('PostgreSQL 16.8 on x86_64-pc-linux-gnu, compiled by Debian clang version 12.0.1, 64-bit',)


In [1]:
!curl ifconfig.me

34.139.51.222

In [9]:
municipios_ibge_tse = pd.read_csv('/content/municipios_ibge_tse.csv', encoding="utf-8", sep = ',')

In [10]:
municipios_ibge_tse.to_sql(
    name="municipios_ibge_tse",
    con=engine,
    index=False
)

570

In [12]:
all = pd.DataFrame()

# states = ['AC', 'AL', 'AM', 'AP', 'DF', 'ES', 'MA', 'MS', 'MG', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR', 'SC', 'SE', 'ZZ', 'TO', 'SP', 'RS', 'CE', 'GO', 'BA']
states = ['RS']

for state in states:
  st = pd.read_csv('/content/drive/MyDrive/data/eleicoes2022unziped/bweb_2t_'+state+'_311020221535.csv', encoding="ISO-8859-1", sep = ';',
                   dtype={
                            "SG_UF": pd.CategoricalDtype(),
                            "CD_MUNICIPIO": pd.Int64Dtype(),
                            "NR_LOCAL_VOTACAO": pd.Int64Dtype(),
                            "NR_ZONA": pd.Int64Dtype(),
                            "NR_SECAO": pd.CategoricalDtype()
                        })
  toUnion = [all, st]

  all = pd.concat(toUnion)

all


,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,CD_PLEITO,DT_PLEITO,NR_TURNO,CD_ELEICAO,DS_ELEICAO,SG_UF,CD_MUNICIPIO,NM_MUNICIPIO,NR_ZONA,NR_SECAO,NR_LOCAL_VOTACAO,CD_CARGO_PERGUNTA,DS_CARGO_PERGUNTA,NR_PARTIDO,SG_PARTIDO,NM_PARTIDO,DT_BU_RECEBIDO,QT_APTOS,QT_COMPARECIMENTO,QT_ABSTENCOES,CD_TIPO_URNA,DS_TIPO_URNA,CD_TIPO_VOTAVEL,DS_TIPO_VOTAVEL,NR_VOTAVEL,NM_VOTAVEL,QT_VOTOS,NR_URNA_EFETIVADA,CD_CARGA_1_URNA_EFETIVADA,CD_CARGA_2_URNA_EFETIVADA,CD_FLASHCARD_URNA_EFETIVADA,DT_CARGA_URNA_EFETIVADA,DS_CARGO_PERGUNTA_SECAO,DS_AGREGADAS,DT_ABERTURA,DT_ENCERRAMENTO,QT_ELEITORES_BIOMETRIA_NH,DT_EMISSAO_BU,NR_JUNTA_APURADORA,NR_TURMA_APURADORA
0,31/10/2022,15:51:32,2022,0,Eleição Ordinária,407,30/10/2022,2,545,Eleição Geral Federal 2022,RS,88013,PORTO ALEGRE,1,1,1422,1,Presidente,-1,#NULO#,#NULO#,30/10/2022 18:34:54,355,282,73,1,APURADA,2,Branco,95,Branco,10,2215453,787.689.897.600.495.174.,531.998,A9CA01B1,22/09/2022 14:12:00,1 - 1,#NULO#,30/10/2022 08:00:01,30/10/2022 17:00:50,9,30/10/2022 17:03:19,-1,-1
1,31/10/2022,15:51:32,2022,0,Eleição Ordinária,407,30/10/2022,2,545,Eleição Geral Federal 2022,RS,88013,PORTO ALEGRE,1,1,1422,1,Presidente,22,PL,Partido Liberal,30/10/2022 18:34:54,355,282,73,1,APURADA,1,Nominal,22,JAIR BOLSONARO,108,2215453,787.689.897.600.495.174.,531.998,A9CA01B1,22/09/2022 14:12:00,1 - 1,#NULO#,30/10/2022 08:00:01,30/10/2022 17:00:50,9,30/10/2022 17:03:19,-1,-1
2,31/10/2022,15:51:32,2022,0,Eleição Ordinária,407,30/10/2022,2,545,Eleição Geral Federal 2022,RS,88013,PORTO ALEGRE,1,1,1422,1,Presidente,-1,#NULO#,#NULO#,30/10/2022 18:34:54,355,282,73,1,APURADA,3,Nulo,96,Nulo,9,2215453,787.689.897.600.495.174.,531.998,A9CA01B1,22/09/2022 14:12:00,1 - 1,#NULO#,30/10/2022 08:00:01,30/10/2022 17:00:50,9,30/10/2022 17:03:19,-1,-1
3,31/10/2022,15:51:32,2022,0,Eleição Ordinária,407,30/10/2022,2,545,Eleição Geral Federal 2022,RS,88013,PORTO ALEGRE,1,1,1422,1,Presidente,13,PT,Partido dos Trabalhadores,30/10/2022 18:34:54,355,282,73,1,APURADA,1,Nominal,13,LULA,155,2215453,787.689.897.600.495.174.,531.998,A9CA01B1,22/09/2022 14:12:00,1 - 1,#NULO#,30/10/2022 08:00:01,30/10/2022 17:00:50,9,30/10/2022 17:03:19,-1,-1
4,31/10/2022,15:51:32,2022,0,Eleição Ordinária,407,30/10/2022,2,547,Eleições Gerais Estaduais 2022,RS,88013,PORTO ALEGRE,1,1,1422,3,Governador,22,PL,Partido Liberal,30/10/2022 18:34:54,355,282,73,1,APURADA,1,Nominal,22,ONYX LORENZONI,67,2215453,787.689.897.600.495.174.,531.998,A9CA01B1,22/09/2022 14:12:00,3 - 1,#NULO#,30/10/2022 08:00:01,30/10/2022 17:00:50,9,30/10/2022 17:03:19,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215870,31/10/2022,15:51:32,2022,0,Eleição Ordinária,407,30/10/2022,2,545,Eleição Geral Federal 2022,RS,86835,GRAVATAÍ,173,291,1201,1,Presidente,22,PL,Partido Liberal,30/10/2022 18:38:47,99,82,17,1,APURADA,1,Nominal,22,JAIR BOLSONARO,28,2150683,270.318.196.437.934.243.,593.092,5C334FD1,22/09/2022 15:01:00,1 - 291,#NULO#,30/10/2022 08:00:01,30/10/2022 17:02:42,1,30/10/2022 17:04:27,-1,-1
215871,31/10/2022,15:51:32,2022,0,Eleição Ordinária,407,30/10/2022,2,547,Eleições Gerais Estaduais 2022,RS,86835,GRAVATAÍ,173,291,1201,3,Governador,45,PSDB,Partido da Social Democracia Brasileira,30/10/2022 18:38:47,99,82,17,1,APURADA,1,Nominal,45,EDUARDO LEITE,54,2150683,270.318.196.437.934.243.,593.092,5C334FD1,22/09/2022 15:01:00,3 - 291,#NULO#,30/10/2022 08:00:01,30/10/2022 17:02:42,1,30/10/2022 17:04:27,-1,-1
215872,31/10/2022,15:51:32,2022,0,Eleição Ordinária,407,30/10/2022,2,547,Eleições Gerais Estaduais 2022,RS,86835,GRAVATAÍ,173,291,1201,3,Governador,-1,#NULO#,#NULO#,30/10/2022 18:38:47,99,82,17,1,APURADA,2,Branco,95,Branco,5,2150683,270.318.196.437.934.243.,593.092,5C334FD1,22/09/2022 15:01:00,3 - 291,#NULO#,30/10/2022 08:00:01,30/10/2022 17:02:42,1,30/10/2022 17:04:27,-1,-1
215873,31/10/2022,15:51:32,2022,0,Eleição Ordinária,407,30/10/2022,2,547,El

In [13]:
all

,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,CD_PLEITO,DT_PLEITO,NR_TURNO,CD_ELEICAO,DS_ELEICAO,SG_UF,CD_MUNICIPIO,NM_MUNICIPIO,NR_ZONA,NR_SECAO,NR_LOCAL_VOTACAO,CD_CARGO_PERGUNTA,DS_CARGO_PERGUNTA,NR_PARTIDO,SG_PARTIDO,NM_PARTIDO,DT_BU_RECEBIDO,QT_APTOS,QT_COMPARECIMENTO,QT_ABSTENCOES,CD_TIPO_URNA,DS_TIPO_URNA,CD_TIPO_VOTAVEL,DS_TIPO_VOTAVEL,NR_VOTAVEL,NM_VOTAVEL,QT_VOTOS,NR_URNA_EFETIVADA,CD_CARGA_1_URNA_EFETIVADA,CD_CARGA_2_URNA_EFETIVADA,CD_FLASHCARD_URNA_EFETIVADA,DT_CARGA_URNA_EFETIVADA,DS_CARGO_PERGUNTA_SECAO,DS_AGREGADAS,DT_ABERTURA,DT_ENCERRAMENTO,QT_ELEITORES_BIOMETRIA_NH,DT_EMISSAO_BU,NR_JUNTA_APURADORA,NR_TURMA_APURADORA
0,31/10/2022,15:51:32,2022,0,Eleição Ordinária,407,30/10/2022,2,545,Eleição Geral Federal 2022,RS,88013,PORTO ALEGRE,1,1,1422,1,Presidente,-1,#NULO#,#NULO#,30/10/2022 18:34:54,355,282,73,1,APURADA,2,Branco,95,Branco,10,2215453,787.689.897.600.495.174.,531.998,A9CA01B1,22/09/2022 14:12:00,1 - 1,#NULO#,30/10/2022 08:00:01,30/10/2022 17:00:50,9,30/10/2022 17:03:19,-1,-1
1,31/10/2022,15:51:32,2022,0,Eleição Ordinária,407,30/10/2022,2,545,Eleição Geral Federal 2022,RS,88013,PORTO ALEGRE,1,1,1422,1,Presidente,22,PL,Partido Liberal,30/10/2022 18:34:54,355,282,73,1,APURADA,1,Nominal,22,JAIR BOLSONARO,108,2215453,787.689.897.600.495.174.,531.998,A9CA01B1,22/09/2022 14:12:00,1 - 1,#NULO#,30/10/2022 08:00:01,30/10/2022 17:00:50,9,30/10/2022 17:03:19,-1,-1
2,31/10/2022,15:51:32,2022,0,Eleição Ordinária,407,30/10/2022,2,545,Eleição Geral Federal 2022,RS,88013,PORTO ALEGRE,1,1,1422,1,Presidente,-1,#NULO#,#NULO#,30/10/2022 18:34:54,355,282,73,1,APURADA,3,Nulo,96,Nulo,9,2215453,787.689.897.600.495.174.,531.998,A9CA01B1,22/09/2022 14:12:00,1 - 1,#NULO#,30/10/2022 08:00:01,30/10/2022 17:00:50,9,30/10/2022 17:03:19,-1,-1
3,31/10/2022,15:51:32,2022,0,Eleição Ordinária,407,30/10/2022,2,545,Eleição Geral Federal 2022,RS,88013,PORTO ALEGRE,1,1,1422,1,Presidente,13,PT,Partido dos Trabalhadores,30/10/2022 18:34:54,355,282,73,1,APURADA,1,Nominal,13,LULA,155,2215453,787.689.897.600.495.174.,531.998,A9CA01B1,22/09/2022 14:12:00,1 - 1,#NULO#,30/10/2022 08:00:01,30/10/2022 17:00:50,9,30/10/2022 17:03:19,-1,-1
4,31/10/2022,15:51:32,2022,0,Eleição Ordinária,407,30/10/2022,2,547,Eleições Gerais Estaduais 2022,RS,88013,PORTO ALEGRE,1,1,1422,3,Governador,22,PL,Partido Liberal,30/10/2022 18:34:54,355,282,73,1,APURADA,1,Nominal,22,ONYX LORENZONI,67,2215453,787.689.897.600.495.174.,531.998,A9CA01B1,22/09/2022 14:12:00,3 - 1,#NULO#,30/10/2022 08:00:01,30/10/2022 17:00:50,9,30/10/2022 17:03:19,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215870,31/10/2022,15:51:32,2022,0,Eleição Ordinária,407,30/10/2022,2,545,Eleição Geral Federal 2022,RS,86835,GRAVATAÍ,173,291,1201,1,Presidente,22,PL,Partido Liberal,30/10/2022 18:38:47,99,82,17,1,APURADA,1,Nominal,22,JAIR BOLSONARO,28,2150683,270.318.196.437.934.243.,593.092,5C334FD1,22/09/2022 15:01:00,1 - 291,#NULO#,30/10/2022 08:00:01,30/10/2022 17:02:42,1,30/10/2022 17:04:27,-1,-1
215871,31/10/2022,15:51:32,2022,0,Eleição Ordinária,407,30/10/2022,2,547,Eleições Gerais Estaduais 2022,RS,86835,GRAVATAÍ,173,291,1201,3,Governador,45,PSDB,Partido da Social Democracia Brasileira,30/10/2022 18:38:47,99,82,17,1,APURADA,1,Nominal,45,EDUARDO LEITE,54,2150683,270.318.196.437.934.243.,593.092,5C334FD1,22/09/2022 15:01:00,3 - 291,#NULO#,30/10/2022 08:00:01,30/10/2022 17:02:42,1,30/10/2022 17:04:27,-1,-1
215872,31/10/2022,15:51:32,2022,0,Eleição Ordinária,407,30/10/2022,2,547,Eleições Gerais Estaduais 2022,RS,86835,GRAVATAÍ,173,291,1201,3,Governador,-1,#NULO#,#NULO#,30/10/2022 18:38:47,99,82,17,1,APURADA,2,Branco,95,Branco,5,2150683,270.318.196.437.934.243.,593.092,5C334FD1,22/09/2022 15:01:00,3 - 291,#NULO#,30/10/2022 08:00:01,30/10/2022 17:02:42,1,30/10/2022 17:04:27,-1,-1
215873,31/10/2022,15:51:32,2022,0,Eleição Ordinária,407,30/10/2022,2,547,El

In [5]:
all.columns

Index(['DT_GERACAO', 'HH_GERACAO', 'ANO_ELEICAO', 'CD_TIPO_ELEICAO',
       'NM_TIPO_ELEICAO', 'CD_PLEITO', 'DT_PLEITO', 'NR_TURNO', 'CD_ELEICAO',
       'DS_ELEICAO', 'SG_UF', 'CD_MUNICIPIO', 'NM_MUNICIPIO', 'NR_ZONA',
       'NR_SECAO', 'NR_LOCAL_VOTACAO', 'CD_CARGO_PERGUNTA',
       'DS_CARGO_PERGUNTA', 'NR_PARTIDO', 'SG_PARTIDO', 'NM_PARTIDO',
       'DT_BU_RECEBIDO', 'QT_APTOS', 'QT_COMPARECIMENTO', 'QT_ABSTENCOES',
       'CD_TIPO_URNA', 'DS_TIPO_URNA', 'CD_TIPO_VOTAVEL', 'DS_TIPO_VOTAVEL',
       'NR_VOTAVEL', 'NM_VOTAVEL', 'QT_VOTOS', 'NR_URNA_EFETIVADA',
       'CD_CARGA_1_URNA_EFETIVADA', 'CD_CARGA_2_URNA_EFETIVADA',
       'CD_FLASHCARD_URNA_EFETIVADA', 'DT_CARGA_URNA_EFETIVADA',
       'DS_CARGO_PERGUNTA_SECAO', 'DS_AGREGADAS', 'DT_ABERTURA',
       'DT_ENCERRAMENTO', 'QT_ELEITORES_BIOMETRIA_NH', 'DT_EMISSAO_BU',
       'NR_JUNTA_APURADORA', 'NR_TURMA_APURADORA'],
      dtype='object')

In [14]:
all.to_sql(
    name="2022_2t_rs",
    con=engine,
    index=False
)

253